# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
2. Encoder, Decoder를 만들고, Seq2Seq(Encoder+Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다. (모델 추론 + while 문)

이번주 목요일 -> 코드와 실행결과 보여주기 발표 

1. 데이터 로드

In [1]:
# %pip install torchviz -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd 

df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv')
df = df[['Q', 'A']]
df


,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


In [2]:
print(df.shape)
df.head(10)

(11823, 2)


,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
5,SD카드 망가졌어,다시 새로 사는 게 마음 편해요.
6,SD카드 안돼,다시 새로 사는 게 마음 편해요.
7,SNS 맞팔 왜 안하지ㅠㅠ,잘 모르고 있을 수도 있어요.
8,SNS 시간낭비인 거 아는데 매일 하는 중,시간을 정하고 해보세요.
9,SNS 시간낭비인데 자꾸 보게됨,시간을 정하고 해보세요.


In [3]:
# 학습시간 단축을 위해 랜덤 샘플링 (5000개) 진행
df = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(df.shape)

# 빈 데이터 제거
df = df.dropna().reset_index(drop=True)
print(df.shape)


(5000, 2)
(5000, 2)


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


2. 데이터 전처리

In [5]:
# 전역변수 선언
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 8000
EMBEDDING_DIM = 100
LATENT_DIM = 128

In [ ]:
SOS_TOKEN = '<sos>'  # SOS/EOS 토큰 추가
EOS_TOKEN = '<eos>'

decoder_inputs  = [SOS_TOKEN + ' ' + a for a in df['A']]
decoder_targets = [a + ' ' + EOS_TOKEN  for a in df['A']]

print('Q 예시:             ', df['Q'][0])
print('A 예시:             ', df['A'][0])
print('decoder_input 예시: ', decoder_inputs[0])
print('decoder_target 예시:', decoder_targets[0])

Q 예시:              죽을거 같네
A 예시:              나쁜 생각 하지 마세요.
decoder_input 예시:  <sos> 나쁜 생각 하지 마세요.
decoder_target 예시: 나쁜 생각 하지 마세요. <eos>


토큰화 & 패딩 처리

In [ ]:

# 단어사전 생성

tokenizer = Tokenizer(num_words=MAX_VOCAB, filters='')
tokenizer.fit_on_texts(df['Q'].tolist() + decoder_inputs + decoder_targets)

print(f'전체 단어 수: {len(tokenizer.word_index)}')
print(f'사용 단어 수: {MAX_VOCAB}')

전체 단어 수: 12967
사용 단어 수: 8000


토큰화 & 패딩 & 텐서변환

In [11]:
# 텍스트 → 정수 시퀀스 → 패딩
MAX_LEN = 20

encoder_seqs = tokenizer.texts_to_sequences(df['Q'])
dec_in_seqs  = tokenizer.texts_to_sequences(decoder_inputs)
dec_tgt_seqs = tokenizer.texts_to_sequences(decoder_targets)

# padding='post': 짧은 문장 뒤에 0 채움
encoder_padded = pad_sequences(encoder_seqs, maxlen=MAX_LEN, padding='post', truncating='post')
dec_in_padded  = pad_sequences(dec_in_seqs,  maxlen=MAX_LEN, padding='post', truncating='post')
dec_tgt_padded = pad_sequences(dec_tgt_seqs, maxlen=MAX_LEN, padding='post', truncating='post')

# PyTorch 텐서로 변환
X_enc = torch.tensor(encoder_padded, dtype=torch.long)
X_dec = torch.tensor(dec_in_padded,  dtype=torch.long)
y_dec = torch.tensor(dec_tgt_padded, dtype=torch.long)

print('인코더 입력 shape:', X_enc.shape)  # (샘플 수, MAX_LEN)
print('디코더 입력 shape:', X_dec.shape)
print('디코더 타겟 shape:', y_dec.shape)

인코더 입력 shape: torch.Size([5000, 20])
디코더 입력 shape: torch.Size([5000, 20])
디코더 타겟 shape: torch.Size([5000, 20])
